# Day 15 - Hyperparameter Tuning (KNN)
Using: global_supply_chain_risk_2026.csv

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [4]:
df = pd.read_csv("global_supply_chain_risk_2026.csv")
df.head()

,Shipment_ID,Date,Origin_Port,Destination_Port,Transport_Mode,Product_Category,Distance_km,Weight_MT,Fuel_Price_Index,Geopolitical_Risk_Score,Weather_Condition,Carrier_Reliability_Score,Lead_Time_Days,Disruption_Occurred
0,SC-10000,2025-10-16,Singapore,Los Angeles,Rail,Textiles,5930.83,197.42,2.43,5.0,Hurricane,0.865,41.39,1
1,SC-10001,2024-04-24,Singapore,Shanghai,Rail,Automotive,14285.36,237.24,2.30,7.5,Storm,0.592,40.92,1
2,SC-10002,2024-01-26,Rotterdam,Los Angeles,Rail,Perishables,11113.91,427.42,1.78,5.6,Rain,0.673,11.54,0
3,SC-10003,2024-10-08,Busan,Hamburg,Rail,Electronics,9180.55,170.66,3.20,0.8,Hurricane,0.832,53.13,1
4,SC-10004,2024-09-07,Busan,Singapore,Air,Perishables,2762.27,434.96,2.77,1.9,Fog,0.741,0.50,1


In [5]:
df = df.dropna()

le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = le.fit_transform(df[col])

X = df.drop("Disruption_Occurred", axis=1)
y = df["Disruption_Occurred"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
# Default KNN
default_knn = KNeighborsClassifier()

default_knn.fit(X_train, y_train)

default_pred = default_knn.predict(X_test)

default_accuracy = accuracy_score(y_test, default_pred)

print("Default Model Accuracy:", default_accuracy)

Default Model Accuracy: 0.539


In [7]:
# GridSearchCV

param_grid = {
    "n_neighbors":[3,5,7,9,11,13]
}

grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train,y_train)

print("Best Parameters:",grid.best_params_)
print("Best Cross Validation Score:",grid.best_score_)

Best Parameters: {'n_neighbors': 11}
Best Cross Validation Score: 0.5619999999999999


In [8]:
best_knn = grid.best_estimator_

grid_pred = best_knn.predict(X_test)

grid_accuracy = accuracy_score(y_test,grid_pred)

print("GridSearch Accuracy:",grid_accuracy)

GridSearch Accuracy: 0.557


In [9]:
# RandomizedSearchCV

random = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_distributions=param_grid,
    n_iter=6,
    cv=5,
    random_state=42,
    scoring="accuracy"
)

random.fit(X_train,y_train)

print("Best Parameters:",random.best_params_)
print("Best Cross Validation Score:",random.best_score_)

Best Parameters: {'n_neighbors': 11}
Best Cross Validation Score: 0.5619999999999999


In [10]:
random_knn = random.best_estimator_

random_pred = random_knn.predict(X_test)

random_accuracy = accuracy_score(y_test,random_pred)

print("RandomizedSearch Accuracy:",random_accuracy)

RandomizedSearch Accuracy: 0.557


In [11]:
comparison = pd.DataFrame({
    "Method":["Default Model","GridSearchCV","RandomizedSearchCV"],
    "Accuracy":[default_accuracy,grid_accuracy,random_accuracy]
})

print(comparison)

best_method = comparison.loc[comparison["Accuracy"].idxmax()]

print("\nBest Method:",best_method["Method"])
print("Best Accuracy:",best_method["Accuracy"])

print("\nConclusion:")
print("The best method achieved the highest accuracy after selecting the optimal value of n_neighbors using hyperparameter tuning.")

               Method  Accuracy
0       Default Model     0.539
1        GridSearchCV     0.557
2  RandomizedSearchCV     0.557

Best Method: GridSearchCV
Best Accuracy: 0.557

Conclusion:
The best method achieved the highest accuracy after selecting the optimal value of n_neighbors using hyperparameter tuning.



# Theory Answers

### 1. What is Hyperparameter Tuning?
Hyperparameter tuning is the process of finding the best settings for a machine learning model to improve its performance.

### 2. Difference between Parameters and Hyperparameters?
- Parameters are learned from data during training.
- Hyperparameters are set before training and control the learning process.

### 3. What is GridSearchCV?
GridSearchCV tests every possible combination of hyperparameters and selects the best one using cross-validation.

### 4. What is RandomizedSearchCV?
RandomizedSearchCV randomly selects combinations of hyperparameters and evaluates them.

### 5. Difference between GridSearchCV and RandomizedSearchCV?
GridSearchCV checks all combinations, while RandomizedSearchCV checks only random combinations, making it faster.

### 6. Why is Cross Validation used?
Cross-validation helps evaluate the model on different data splits, reducing overfitting and improving reliability.

### 7. Purpose of best_params_ and best_score_?
- **best_params_** returns the best hyperparameter values.
- **best_score_** returns the highest cross-validation accuracy obtained.
